# silver_faturamento

Camada **Silver** do faturamento: `faturas`.

- **Entrada:** `bronze.tb_faturas` (referência: `silver.tb_unidades_consumidoras`)
- **Saída:** `silver.tb_faturas` e `silver.tb_faturas_quarentena`
- **Segue o Padrão de Limpeza:** ler e contar → estrutura → tipos → domínios → duplicatas → consistência → integridade → checagens e reconciliação → gravar.
- **Alimenta:** `gold_receita`, `gold_kpis` e `gold_visao_360`.
- Regras levantadas no notebook `exploracao_faturamento`.

**Contrato da tabela** (não mude nome nem tipo sem combinar com quem usa na Gold):

| Coluna | Tipo | Regra |
|---|---|---|
| `id_fatura` | string | chave primária |
| `id_uc` | string | existe em `silver.tb_unidades_consumidoras`; (`id_uc`, `mes_referencia`) também é única |
| `mes_referencia` | string `yyyy-MM` | competência, no mesmo formato da `tb_leituras` (join direto) |
| `data_emissao`, `data_vencimento` | date | hoje o vencimento é sempre emissão + 12 dias |
| `data_pagamento` | date | nulo = não paga (inclui a sentinela `9999-12-31`) |
| `valor_energia`, `tributos`, `valor_total` | decimal(14,2) | total = energia + tributos |
| `bandeira_tarifaria` | string | VERDE, AMARELA, VERMELHA P1, VERMELHA P2 |
| `status` | string | PAGA (com data de pagamento) ou EM ABERTO (sem) |
| `flag_*` | boolean | consistências do passo 6 (hoje todas `false`) |

In [0]:
%run ./funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")
NOTEBOOK = "silver_faturamento"

In [0]:
# ---------- Regras de negócio: Faturas ----------
DATAS = ["data_emissao", "data_vencimento", "data_pagamento"]    # 4 formatos + sentinela 9999-12-31: o parse_data trata
VALORES = ["valor_energia", "tributos", "valor_total"]            # "76.93" e "R$ 1.234,56": o parse_decimal trata
BANDEIRAS = ["VERDE", "AMARELA", "VERMELHA P1", "VERMELHA P2"]
STATUS = ["PAGA", "EM ABERTO"]

In [0]:
# Passo 1: ler e contar
ft = ler_bronze(CATALOGO, "faturas")
N_BRONZE = ft.count()
COLUNAS = [c for c in ft.columns if not c.startswith("_")]

# Passo 2: estrutura (aparar hoje não acha espaços: proteção)
ft = aparar_textos(remover_cabecalho_repetido(ft))
REMOVIDAS = {"cabecalho": N_BRONZE - ft.count()}      # o que cada passo remover, medido
registrar("faturas", "cabeçalho do CSV repetido no meio do arquivo",
          "remover_cabecalho_repetido", "REMOVIDO", REMOVIDAS["cabecalho"])

In [0]:
# Conta numa única leitura o que os passos 3 e 4 vão corrigir
n = ft.agg(
    F.sum((~F.col("data_emissao").rlike(r"^\d{4}-\d{2}-\d{2}$")).cast("int")).alias("datas"),
    F.sum((F.col("data_pagamento") == "9999-12-31").cast("int")).alias("sentinela"),
    F.sum(F.concat_ws(" ", *VALORES).contains("R$").cast("int")).alias("moeda"),
    F.sum((F.col("bandeira_tarifaria") != padronizar_texto("bandeira_tarifaria")).cast("int")).alias("bandeira"),
).first()

# Passo 3: tipos. O checar_parse falha se algum valor se perder (a sentinela não conta como perda)
checar_parse(ft, DATAS, parse_data, "faturas")
checar_parse(ft, VALORES, parse_decimal, "faturas")
checar_parse(ft, ["mes_referencia"], parse_data, "faturas", formatos=("yyyy-MM",))   # só valida: continua texto
ft = ft.withColumns({c: parse_data(c) for c in DATAS} | {c: parse_decimal(c) for c in VALORES})

# Passo 4: domínios (status já vem limpo: padronizado como proteção)
ft = ft.withColumns({c: padronizar_texto(c) for c in ["bandeira_tarifaria", "status"]})

registrar("faturas", "data_emissao em 4 formatos (yyyy-MM-dd, dd/MM/yyyy, dd.MM.yyyy, MM-dd-yyyy)",
          "parse_data (hífen = mês-dia-ano: vencimento = emissão + 12 dias em 100%)", "CORRIGIDO", n["datas"])
registrar("faturas", "data_pagamento 9999-12-31 em faturas EM ABERTO (sentinela de 'sem pagamento', não atraso)",
          "parse_data (sentinela -> nulo); status mantido", "CORRIGIDO", n["sentinela"])
registrar("faturas", "valor_total no formato 'R$ 1.234,56'", "parse_decimal", "CORRIGIDO", n["moeda"])
registrar("faturas", "bandeira_tarifaria em caixa mista (Verde, Vermelha P1...)", "padronizar_texto",
          "CORRIGIDO", n["bandeira"])

In [0]:
# Passo 5: duplicatas idênticas (hoje 0: proteção para as cargas mensais)
n_antes = N_BRONZE - REMOVIDAS["cabecalho"]
ft = deduplicar(ft)
REMOVIDAS["duplicatas"] = n_antes - ft.count()
registrar("faturas", "linhas duplicadas idênticas", "deduplicar(df)", "REMOVIDO", REMOVIDAS["duplicatas"])

# Passo 6: consistência (hoje 100% respeitada; violação vira flag)
FLAGS = {
    "flag_total_inconsistente": (
        F.col("valor_total") != F.col("valor_energia") + F.col("tributos"),
        "valor_total diferente de valor_energia + tributos"),
    "flag_status_pagamento_inconsistente": (
        (F.col("status") == "PAGA") != F.col("data_pagamento").isNotNull(),
        "status incompatível com data_pagamento (PAGA sem data ou EM ABERTO com data)"),
}
for flag, (condicao, _) in FLAGS.items():
    ft = criar_flag(ft, flag, condicao)
contagem = ft.agg(*[F.sum(F.col(f).cast("int")).alias(f) for f in FLAGS]).first()
for flag, (_, problema) in FLAGS.items():
    registrar("faturas", problema, flag, "FLAG", contagem[flag])

In [0]:
# Passo 7: integridade (por último). As faturas FTX… são cópias de faturas reais com id_uc inexistente
uc = spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras")
ft, ft_quarentena = separar_orfaos(ft, uc, "id_uc", motivo="id_uc inexistente em silver.tb_unidades_consumidoras")
N_QUARENTENA = ft_quarentena.count()
registrar("faturas", "fatura de UC inexistente (ids FTX…, cópias de faturas reais; inclui os pares UC+mês repetidos)",
          "separar_orfaos vs silver.tb_unidades_consumidoras", "QUARENTENA", N_QUARENTENA)

In [0]:
# Passo 8: checagens e reconciliação. Se algo falhar, o notebook para ANTES de gravar
checar_unicidade(ft, ["id_fatura"], "faturas")
checar_unicidade(ft, ["id_uc", "mes_referencia"], "faturas")
checar_sem_nulos(ft, [c for c in COLUNAS if c != "data_pagamento"], "faturas")   # pagamento nulo = não paga
checar_dominio(ft, "bandeira_tarifaria", BANDEIRAS, "faturas")
checar_dominio(ft, "status", STATUS, "faturas")
reconciliar("faturas", N_BRONZE, ft.count(), N_QUARENTENA, REMOVIDAS)

In [0]:
# Passo 9: gravar (ordem: negócio -> flags -> rastreio) e log de qualidade
ft = ft.select(*COLUNAS, *FLAGS, "_source_file", "_ingestion_timestamp")
gravar_silver(ft, CATALOGO, "faturas", quarentena=ft_quarentena)
gravar_log(CATALOGO, NOTEBOOK)

display(spark.table(f"{CATALOGO}.silver.tb_log_qualidade")
             .filter(F.col("notebook") == NOTEBOOK).orderBy("tabela", "destino"))